# Objective Track: MiniCPM-V 4.6 (age, gender, ethnicity)

Run order (Colab):
1. Run sections 1 to 6. Section 5 asks the model (2,880 images x 3 questions = 8,640 queries) and can be stopped and resumed. Section 6 writes the unclear answers to text files.
2. Send the files `unclear_*.jsonl` together with the matcher instructions printed at the end of section 6 to the matcher. Put the matcher's returned files in `matcher_returns/`.
3. Run sections 7 to 11.

Decisions fixed in this notebook:
- "none" (no option chosen) counts as **wrong**.
- McNemar's test is the **exact (binomial)** version, with **no multiple-comparison correction**.
- The letter rule, the question wording and the matcher file format are our own design choices (no source).

## 1. Setup, paths, load data

In [1]:
# [section 1a: setup]
import os, re, json, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import binomtest

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# [section 1b: config]
MAIN_DIR = '/content/drive/MyDrive/Frugal AI Project/'
OUTPUT_DIR = os.path.join(MAIN_DIR, 'Objective Track')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS_PATH = os.path.join(MAIN_DIR, 'sampled_dataset.csv')
BASE_DIR    = os.path.join(MAIN_DIR, 'generated_portraits_hashed_final/generated_portraits_hashed')
CAP_DIR     = os.path.join(MAIN_DIR, 'generated_cap')
GLASSES_DIR = os.path.join(MAIN_DIR, 'generated_glasses')
MASK_DIR    = os.path.join(MAIN_DIR, 'generated_masks')

IMAGE_TYPES = ['base', 'cap', 'glasses', 'mask']
ACCESSORIES = ['cap', 'glasses', 'mask']
ATTRIBUTES  = ['ethnicity', 'gender', 'age']

RAW_PATH = os.path.join(OUTPUT_DIR, 'raw_answers.jsonl')
MATCHER_RETURNS_DIR = os.path.join(OUTPUT_DIR, 'matcher_returns')
os.makedirs(MATCHER_RETURNS_DIR, exist_ok=True)

In [3]:

# [section 1c: load labels and check image files]
REQUIRED = ['HASH', 'ETHNICITY', 'GENDER', 'AGE']
df = pd.read_csv(LABELS_PATH)
assert set(REQUIRED) <= set(df.columns), f"missing columns, found {list(df.columns)}"
df = df[REQUIRED].copy()
assert df['HASH'].is_unique, "duplicate HASH values"
print("identities:", len(df))

def image_path(hash_, image_type):
    if image_type == 'base':    return f"{BASE_DIR}/{hash_}.jpg"
    if image_type == 'cap':     return f"{CAP_DIR}/{hash_}_hat.png"
    if image_type == 'glasses': return f"{GLASSES_DIR}/{hash_}_glasses.png"
    if image_type == 'mask':    return f"{MASK_DIR}/{hash_}_mask.png"
    raise ValueError(image_type)

missing = [(h, t) for h in df['HASH'] for t in IMAGE_TYPES if not os.path.exists(image_path(h, t))]
assert not missing, f"{len(missing)} image files not found, first ones: {missing[:3]}"
print("all", len(df) * len(IMAGE_TYPES), "image files found")

identities: 720
all 2880 image files found


## 2. Questions
One single-choice question per attribute. Same wording and same option order on every image. Option order follows the order in the data.

In [4]:
# [section 2: questions]
OPTIONS = {
    'ethnicity': ['African', 'Asian', 'Caucasian', 'Indian'],
    'gender':    ['Female', 'Male'],
    'age':       ['25', '40', '55'],
}
QUESTION_STEM = {
    'ethnicity': "What is the ethnicity of the person in this image?",
    'gender':    "What is the gender of the person in this image?",
    'age':       "What is the age of the person in this image?",
}
LETTERS = 'ABCD'

def valid_letters(attribute):
    return list(LETTERS[:len(OPTIONS[attribute])])

def question_text(attribute):
    lines = [QUESTION_STEM[attribute]]
    for letter, option in zip(LETTERS, OPTIONS[attribute]):
        lines.append(f"{letter}. {option}")
    lines.append("Answer with only the letter.")
    return "\n".join(lines)

# the true labels in the data must all be among the options
for attribute in ATTRIBUTES:
    labels = set(df[attribute.upper()].astype(str))
    assert labels <= set(OPTIONS[attribute]), f"{attribute}: labels {labels} not all in options"

for attribute in ATTRIBUTES:
    print(question_text(attribute), "\n")

What is the ethnicity of the person in this image?
A. African
B. Asian
C. Caucasian
D. Indian
Answer with only the letter. 

What is the gender of the person in this image?
A. Female
B. Male
Answer with only the letter. 

What is the age of the person in this image?
A. 25
B. 40
C. 55
Answer with only the letter. 



## 3. Model and the function that asks one question on one image
The loading and generation calls, including `downsample_mode="4x"`, `max_slice_nums=36` and `max_new_tokens=100`, are copied from Nour's pilot notebook. They have not been checked against the model card, and no sampling setting is passed to `generate`, so the model's own default generation config applies.

In [5]:
# [section 3: model]
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "openbmb/MiniCPM-V-4.6"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)

def ask(image_path, text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path},
                {"type": "text", "text": text},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        downsample_mode="4x",
        max_slice_nums=36,
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        downsample_mode="4x",
        max_new_tokens=100,
    )
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

preprocessor_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.60GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/779 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

## 4. Letter rule
Reads the raw answer and returns the chosen letter, or `None` (unclear) when it does not clearly commit to exactly one valid letter. The rule is deliberately strict: anything doubtful goes to the matcher instead of being graded directly.

Accepted: the letter alone, with optional quotes, brackets or bold (`B`, `(B)`, `**B**`), an optional `Answer:` prefix, or the letter followed by `.` `,` `:` `;` `)` and an explanation (`B, because ...`).
Not accepted: a letter followed by a word (`A person ...`), a letter outside the valid options, or an explanation that itself contains another valid letter as a standalone capital (`A, B or C`).

In [6]:
# [section 4: letter rule]
_PREFIX = re.compile(r'^(?i:answer)\s*[:\-]?\s*')
_LETTER = re.compile(r'''^[(\[*"'`]*([A-Z])(?:[)\]*"'`]+|(?=[.,:;]|$))''')

def extract_letter(raw, valid):
    if raw is None:
        return None
    s = _PREFIX.sub('', raw.strip())
    m = _LETTER.match(s)
    if not m:
        return None
    letter = m.group(1)
    if letter not in valid:
        return None
    rest = s[m.end():]
    # reject if the explanation contains another valid letter as a standalone capital
    for other in valid:
        if other != letter and re.search(rf'(?<![A-Za-z]){other}(?![A-Za-z])', rest):
            return None
    return letter

# quick self-check of the rule on made-up strings (not model output)
_tests = [
    ("B", ['A','B','C'], 'B'), ("B.", ['A','B','C'], 'B'), ("(B)", ['A','B','C'], 'B'),
    ("**B**", ['A','B','C'], 'B'), ("Answer: C", ['A','B','C'], 'C'),
    ("B, because the person looks middle-aged", ['A','B','C'], 'B'),
    ("C) 55", ['A','B','C'], 'C'),
    ("A person in their 40s", ['A','B','C'], None), ("A, B or C", ['A','B','C'], None),
    ("I can't determine age from a photo.", ['A','B','C'], None),
    ("D", ['A','B'], None), ("", ['A','B','C'], None),
]
for raw, valid, expected in _tests:
    got = extract_letter(raw, valid)
    assert got == expected, f"rule failed on {raw!r}: got {got}, expected {expected}"
print("letter rule self-check passed on", len(_tests), "strings")

letter rule self-check passed on 12 strings


## 5. Ask the model (raw answers only)
Every raw answer is appended to `raw_answers.jsonl` as soon as it is produced. If the run stops, run this cell again: finished (identity, image, attribute) combinations are skipped. Parsing happens later, so changing the letter rule never requires re-asking the model.

In [7]:
# [section 5: run]
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

done = set()
if os.path.exists(RAW_PATH):
    done = {(r['hash'], r['image_type'], r['attribute']) for r in load_jsonl(RAW_PATH)}
print("already done:", len(done), "of", len(df) * len(IMAGE_TYPES) * len(ATTRIBUTES))

with open(RAW_PATH, 'a', encoding='utf-8') as f:
    for hash_ in tqdm(df['HASH'], desc="identities"):
        for image_type in IMAGE_TYPES:
            for attribute in ATTRIBUTES:
                if (hash_, image_type, attribute) in done:
                    continue
                raw_answer = ask(image_path(hash_, image_type), question_text(attribute))
                f.write(json.dumps({
                    'hash': hash_, 'image_type': image_type,
                    'attribute': attribute, 'raw_answer': raw_answer,
                }, ensure_ascii=False) + '\n')
                f.flush()

already done: 8640 of 8640


identities:   0%|          | 0/720 [00:00<?, ?it/s]

## 6. Apply the letter rule and write the unclear answers to text files
One file per attribute x image type (12 files). Each line holds the id, the attribute, the image type, the options and the raw answer. The true label is not included.

In [8]:
# [section 6: parse and write unclear files]
raw = pd.DataFrame(load_jsonl(RAW_PATH))
expected_rows = len(df) * len(IMAGE_TYPES) * len(ATTRIBUTES)
assert len(raw) == expected_rows, f"expected {expected_rows} raw answers, found {len(raw)}"
assert not raw.duplicated(['hash', 'image_type', 'attribute']).any(), "duplicate raw answers"

raw['id'] = raw['hash'] + '|' + raw['image_type'] + '|' + raw['attribute']
raw['rule_letter'] = [
    extract_letter(a, valid_letters(attr)) for a, attr in zip(raw['raw_answer'], raw['attribute'])
]

unclear = raw[raw['rule_letter'].isna()]
for old in [p for p in os.listdir(OUTPUT_DIR) if p.startswith('unclear_')]:
    os.remove(os.path.join(OUTPUT_DIR, old))

for (attribute, image_type), g in unclear.groupby(['attribute', 'image_type']):
    options = {L: o for L, o in zip(LETTERS, OPTIONS[attribute])}
    path = os.path.join(OUTPUT_DIR, f"unclear_{attribute}_{image_type}.jsonl")
    with open(path, 'w', encoding='utf-8') as f:
        for r in g.itertuples():
            f.write(json.dumps({
                'id': r.id, 'attribute': attribute, 'image_type': image_type,
                'options': options, 'raw_answer': r.raw_answer,
            }, ensure_ascii=False) + '\n')

print("unclear answers:", len(unclear), "of", len(raw))
print(unclear.groupby(['attribute', 'image_type']).size().rename('n_unclear').to_string())

MATCHER_INSTRUCTIONS = (
    "Each line of the attached file(s) is a JSON object with an id, the answer options, and a model's raw answer to a "
    "single-choice question about a face photo. For each line, return the letter of the option that the raw answer "
    "clearly commits to. Return NONE if the answer refuses, hedges between several options, gives a value outside the "
    "options, or does not answer the question. Do not guess a letter for NONE cases. Return one JSON object per line, "
    "in the same order, with exactly the keys id and letter, for example: {\"id\": \"...\", \"letter\": \"B\"}. "
    "Return one file per input file."
)
print("\nMATCHER INSTRUCTIONS:\n" + MATCHER_INSTRUCTIONS)

unclear answers: 0 of 8640
Series([], )

MATCHER INSTRUCTIONS:
Each line of the attached file(s) is a JSON object with an id, the answer options, and a model's raw answer to a single-choice question about a face photo. For each line, return the letter of the option that the raw answer clearly commits to. Return NONE if the answer refuses, hedges between several options, gives a value outside the options, or does not answer the question. Do not guess a letter for NONE cases. Return one JSON object per line, in the same order, with exactly the keys id and letter, for example: {"id": "...", "letter": "B"}. Return one file per input file.


## 7. Load the matcher's answers
Put the returned `.jsonl` files in `Objective Track/matcher_returns/`. This cell stops if any unclear answer has no matcher result, or if a returned letter is not a valid option or `NONE`.

In [9]:
# [section 7: load matcher returns]
matcher = {}
for name in sorted(os.listdir(MATCHER_RETURNS_DIR)):
    if name.endswith('.jsonl'):
        for r in load_jsonl(os.path.join(MATCHER_RETURNS_DIR, name)):
            matcher[r['id']] = str(r['letter']).strip().upper()

unclear_ids = set(raw.loc[raw['rule_letter'].isna(), 'id'])
not_returned = unclear_ids - set(matcher)
assert not not_returned, f"{len(not_returned)} unclear answers have no matcher result, e.g. {sorted(not_returned)[:3]}"
unknown_ids = set(matcher) - unclear_ids
assert not unknown_ids, f"{len(unknown_ids)} matcher ids do not match any unclear answer, e.g. {sorted(unknown_ids)[:3]}"

for id_, letter in matcher.items():
    attribute = id_.split('|')[2]
    assert letter in valid_letters(attribute) + ['NONE'], f"invalid matcher letter {letter!r} for {id_}"

raw['final_letter'] = [
    matcher[r.id] if pd.isna(r.rule_letter) else r.rule_letter for r in raw.itertuples()
]
assert raw['final_letter'].notna().all(), "some answers have no final letter"
raw['source'] = np.where(raw['rule_letter'].isna(), 'matcher', 'rule')
print(raw['source'].value_counts().to_string())

source
rule    8640


## 8. Grade
Correct when the final letter equals the letter of the true label. `NONE` counts as wrong.

In [10]:
# [section 8: grade]
true_letter = {
    attribute: {opt: L for L, opt in zip(LETTERS, OPTIONS[attribute])} for attribute in ATTRIBUTES
}
labels = df.set_index('HASH')

graded = raw.copy()
graded['true_label'] = [
    str(labels.loc[h, attr.upper()]) for h, attr in zip(graded['hash'], graded['attribute'])
]
graded['true_letter'] = [
    true_letter[attr][lab] for attr, lab in zip(graded['attribute'], graded['true_label'])
]
graded['correct'] = (graded['final_letter'] == graded['true_letter'])
graded = graded[['hash', 'attribute', 'image_type', 'raw_answer', 'rule_letter', 'final_letter',
                 'source', 'true_label', 'true_letter', 'correct']]
graded.to_csv(os.path.join(OUTPUT_DIR, 'graded_results.csv'), index=False)
graded.head()

,hash,attribute,image_type,raw_answer,rule_letter,final_letter,source,true_label,true_letter,correct
0,09534c3a20eaad55,ethnicity,base,A,A,A,rule,African,A,True
1,09534c3a20eaad55,gender,base,A,A,A,rule,Female,A,True
2,09534c3a20eaad55,age,base,A,A,A,rule,25,A,True
3,09534c3a20eaad55,ethnicity,cap,A,A,A,rule,African,A,True
4,09534c3a20eaad55,gender,cap,A,A,A,rule,Female,A,True


## 9. Checks
- Base accuracy per attribute (the reference for every accessory result).
- Unclear rate per attribute x image type: share of answers the letter rule could not read.
- None rate per attribute x image type: share of answers that ended as `NONE` (counted as wrong).

In [11]:
# [section 9: checks]
base_acc = (graded[graded['image_type'] == 'base']
            .groupby('attribute')['correct'].agg(accuracy='mean', n='size').reindex(ATTRIBUTES))
base_acc['accuracy'] = (base_acc['accuracy'] * 100).round(2)
print("Base accuracy (%):"); print(base_acc.to_string())

rates = graded.assign(
    unclear=graded['source'].eq('matcher'),
    none=graded['final_letter'].eq('NONE'),
).groupby(['attribute', 'image_type']).agg(
    n=('correct', 'size'), unclear_rate=('unclear', 'mean'), none_rate=('none', 'mean'),
)
rates[['unclear_rate', 'none_rate']] = (rates[['unclear_rate', 'none_rate']] * 100).round(2)
print("\nUnclear and none rates (%):"); print(rates.to_string())

base_acc.to_csv(os.path.join(OUTPUT_DIR, 'check_base_accuracy.csv'))
rates.to_csv(os.path.join(OUTPUT_DIR, 'check_unclear_none_rates.csv'))

Base accuracy (%):
           accuracy    n
attribute               
ethnicity     91.94  720
gender        99.44  720
age           86.53  720

Unclear and none rates (%):
                        n  unclear_rate  none_rate
attribute image_type                              
age       base        720           0.0        0.0
          cap         720           0.0        0.0
          glasses     720           0.0        0.0
          mask        720           0.0        0.0
ethnicity base        720           0.0        0.0
          cap         720           0.0        0.0
          glasses     720           0.0        0.0
          mask        720           0.0        0.0
gender    base        720           0.0        0.0
          cap         720           0.0        0.0
          glasses     720           0.0        0.0
          mask        720           0.0        0.0


## 10. RQ1: exact McNemar test per attribute x accessory
For each identity, compare correct/wrong on the base image with correct/wrong on the accessory image.
- b = correct on base, wrong with accessory.
- c = wrong on base, correct with accessory.
- Accuracy change = accuracy with accessory minus base accuracy = (c - b) / n.
- p-value: exact two-sided binomial test of b out of (b + c) against 0.5. If b + c = 0 there is no flipped identity and the p-value is NaN.

The 9 tests (3 attributes x 3 accessories) are not corrected for multiple comparisons.

In [12]:
# [section 10: McNemar]
def mcnemar_exact(base_correct, accessory_correct):
    base_correct = np.asarray(base_correct, dtype=bool)
    accessory_correct = np.asarray(accessory_correct, dtype=bool)
    n = len(base_correct)
    b = int((base_correct & ~accessory_correct).sum())    # correct -> wrong
    c = int((~base_correct & accessory_correct).sum())    # wrong -> correct
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else float('nan')
    return {
        'n': n,
        'acc_base': round(base_correct.mean() * 100, 2),
        'acc_accessory': round(accessory_correct.mean() * 100, 2),
        'b_correct_to_wrong': b,
        'c_wrong_to_correct': c,
        'acc_change_pts': round((c - b) / n * 100, 2),
        'p_value': p,
    }

def paired_results(graded, hashes=None):
    rows = []
    for attribute in ATTRIBUTES:
        g = graded[graded['attribute'] == attribute]
        if hashes is not None:
            g = g[g['hash'].isin(hashes)]
        wide = g.pivot(index='hash', columns='image_type', values='correct').astype(bool)
        for accessory in ACCESSORIES:
            rows.append({'attribute': attribute, 'accessory': accessory,
                         **mcnemar_exact(wide['base'].values, wide[accessory].values)})
    return pd.DataFrame(rows)

overall = paired_results(graded)
overall.to_csv(os.path.join(OUTPUT_DIR, 'mcnemar_overall.csv'), index=False)
overall

,attribute,accessory,n,acc_base,acc_accessory,b_correct_to_wrong,c_wrong_to_correct,acc_change_pts,p_value
0,ethnicity,cap,720,91.94,85.69,51,6,-6.25,5.676193e-10
1,ethnicity,glasses,720,91.94,90.97,19,12,-0.97,2.810415e-01
2,ethnicity,mask,720,91.94,85.28,52,4,-6.67,1.100806e-11
3,gender,cap,720,99.44,93.33,44,0,-6.11,1.136868e-13
4,gender,glasses,720,99.44,98.89,4,0,-0.56,1.250000e-01
5,gender,mask,720,99.44,98.61,7,1,-0.83,7.031250e-02
6,age,cap,720,86.53,82.50,64,35,-4.03,4.640654e-03
7,age,glasses,720,86.53,84.86,40,28,-1.67,1.818101e-01
8,age,mask,720,86.53,69.31,184,60,-17.22,7.904745e-16


## 11. RQ3: the same test split by ethnicity, gender and age
`n` is the number of identities in the group. The tests are not corrected for multiple comparisons.

In [13]:
# [section 11: groups]
parts = []
for group_var in ['ETHNICITY', 'GENDER', 'AGE']:
    for value in sorted(df[group_var].unique()):
        hashes = set(df.loc[df[group_var] == value, 'HASH'])
        t = paired_results(graded, hashes)
        t.insert(0, 'group_var', group_var)
        t.insert(1, 'group', value)
        parts.append(t)

by_group = pd.concat(parts, ignore_index=True)
by_group.to_csv(os.path.join(OUTPUT_DIR, 'mcnemar_by_group.csv'), index=False)
by_group

,group_var,group,attribute,accessory,n,acc_base,acc_accessory,b_correct_to_wrong,c_wrong_to_correct,acc_change_pts,p_value
0,ETHNICITY,African,ethnicity,cap,180,95.56,94.44,3,1,-1.11,0.625000
1,ETHNICITY,African,ethnicity,glasses,180,95.56,96.11,1,2,0.56,1.000000
2,ETHNICITY,African,ethnicity,mask,180,95.56,92.22,7,1,-3.33,0.070312
3,ETHNICITY,African,gender,cap,180,98.33,89.44,16,0,-8.89,0.000031
4,ETHNICITY,African,gender,glasses,180,98.33,97.22,2,0,-1.11,0.500000
...,...,...,...,...,...,...,...,...,...,...,...
76,AGE,55,gender,glasses,240,99.58,98.75,2,0,-0.83,0.500000
77,AGE,55,gender,mask,240,99.58,97.50,5,0,-2.08,0.062500
78,AGE,55,age,cap,240,95.83,98.75,0,7,2.92,0.015625
79,AGE,55,age,glasses,240,95.83,99.17,0,8,3.33,0.007812
